# Búsqueda anidada de hiperparámetros
El modo **rapido** comprueba el funcionamiento; sus métricas no son resultados finales.
Cambiar `MODO` a `completo` utiliza todas las filas y las configuraciones originales.
Se codifican clases explícitamente, se ajusta la imputación dentro de cada pipeline y se guardan
particiones, métricas y predicciones en una carpeta nueva por ejecución.
Los grupos son **repeticiones**, no participantes. La evaluación externa no interviene en la búsqueda
interna de parámetros, pero elegir la familia mediante esos resultados impide llamarla prueba final
independiente de esa selección. Ver README y `docs/evaluacion.md`.
Las salidas previas se conservan en `reports/historico/`; no son resultados de esta versión.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rehab" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz del repositorio REHAB.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from rehab.data import DATA_DIR

import pandas as pd
from rehab.experiments import run
MODO = "rapido"

In [ ]:
RUN = run(stage="search", mode=MODO)
print("Resultados:", RUN)

In [ ]:
display(pd.read_csv(RUN / "summary.csv"))
display(pd.read_csv(RUN / "fold_metrics.csv"))

Los reportes por clase, matrices de confusión y predicciones están en las subcarpetas por modelo.
La comparación se ordena por F1 macro medio entre folds; la desviación entre folds describe variación,
no un intervalo de confianza. En modo rápido se muestrean repeticiones completas y se reducen los ajustes.

## Modelo para inferencia
Se elige la familia con mayor F1 macro medio externo y se hace una búsqueda adicional por grupos
sobre todos los datos de desarrollo del modo elegido. Se exporta el mejor pipeline reajustado,
con su codificador, columnas, parámetros y versiones. **No es una nueva evaluación independiente**.
En modo rápido este artefacto es solo una prueba. HistGradientBoosting conserva `early_stopping=False`.

In [ ]:
from rehab.experiments import fit_final
MODEL_DIR = fit_final(RUN)
print(MODEL_DIR)